In [1]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import ComplementNB
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from preprocessing.tokenization import *
from pathlib import Path
from sklearn.metrics import f1_score

DATA_DIR = Path("data/final_dataset")
SPLITS_DIR = DATA_DIR / "splits"
TK_DIR = DATA_DIR / "tokenizers"

In [4]:
def make_vectorizer(vocab:str)->TfidfVectorizer:
    with open(vocab, "r") as f:
        tcfg = TokenizerConfig.from_json(f.read())

    tokenizer = Tokenizer(tcfg, False)
    return TfidfVectorizer(
        tokenizer= lambda text: tokenizer.text_to_tokens(text),
        vocabulary=tcfg.tokens.tolist(),
        analyzer="word",
        max_features=10_000,
        token_pattern=None,
        lowercase=False,
        preprocessor=lambda x: x
    )

vectorizer = make_vectorizer(TK_DIR / "top10k.json")
relieable = ["reliable", "political", "clickbait", "bias"]



def extract_df(df : pd.DataFrame, is_train : bool = False):
    print("Categorizing type...")
    y = df["type"].isin(["reliable", "political", "clickbait", "bias"]).to_numpy(dtype=int)

    print("Vectorizing content tokens...")
    if is_train: X_content = vectorizer.fit_transform(df["content"])
    else: X_content = vectorizer.transform(df["content"])

    return X_content, y

In [5]:
train_df = pd.read_csv("data/final_dataset/splits/train.csv")
X_train, y_train = extract_df(train_df, is_train=True)

X_val, y_val = extract_df(pd.read_csv("data/final_dataset/splits/val.csv"))

Categorizing type...
Vectorizing content tokens...
Categorizing type...
Vectorizing content tokens...


In [7]:
from sklearn.linear_model import LogisticRegression

logre_model = LogisticRegression(max_iter=500, C=5, class_weight='balanced')
logre_model.fit(X_train, y_train)

print(f1_score(y_val, logre_model.predict(X_val)))

# 805 -> 823 -> 838 -> 847 ->rel 872

0.8829124835209411


In [8]:
mnb_model = MultinomialNB(alpha=1e-100)

mnb_model.fit(X_train, y_train)

print(f"f1 score: {f1_score(y_val, mnb_model.predict(X_val))}")

f1 score: 0.8607734599992519


In [9]:
cnb_model = ComplementNB(alpha=1e-100)

cnb_model.fit(X_train, y_train)

print(f"f1 score: {f1_score(y_val, cnb_model.predict(X_val))}")

f1 score: 0.8320593244983591


c:\Users\nva\Desktop\GDS2026\.venv\Lib\site-packages\sklearn\naive_bayes.py:1077: RuntimeWarning: divide by zero encountered in log
  logged = np.log(comp_count / comp_count.sum(axis=1, keepdims=True))


In [10]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(penalty='l1', C=1, class_weight='balanced')
svm_model.fit(X_train, y_train)

print(f"f1 score: {f1_score(y_val, svm_model.predict(X_val))}")

# 808 -> 829 -> 841 -> 847 | 852 ->rel 876

f1 score: 0.8858192416284647


In [12]:
import joblib

joblib.dump(svm_model, "data/models/linearSVC.joblib")
#joblib.dump(vectorizer, "data/models/linearSVC_vectorizer.joblib")

['data/models/linearSVC.joblib']

In [10]:
# import json

# with open("data/models/linearSVC.json", 'w') as f:
#     json.dump({
#     'classes' : svm_model.classes_.tolist(),
#     'intercept' : svm_model.intercept_.tolist(),
#     'coef' : svm_model.coef_.tolist(),
# }, f, indent=4)

# vectorizer.get

In [11]:
# model = LinearSVC()
# model.coef_ = svm_model.coef_
# model.intercept_ = svm_model.intercept_
# model.classes_ = svm_model.classes_

# print(f"f1 score: {f1_score(y_val, model.predict(X_val))}")